## Customer Segmentation

### Objective
Group accounts into **behaviorally distinct segments** that business can act on.

This phase clusters accounts into distinct behavioral segments based on transaction activity, cash flow patterns, balance stability, and spending diversity. The objective is to translate financial behavior into actional customer personas.

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load aggregated dataset from notebook_3
df_health = pd.read_csv("../outputs/account_financial_health.csv")

### Feature Selection
Select a compact set of **behavioral features** that:
* Capture meaningful differences in customer behavior
* Are independent of the financial health score
* Can be clearly explained to business stakeholders

The goal is not mathematical optimality, but **actionable segmentation**.

#### Design Principles
1. Behavior over outcomes
    * We avoid using `financial_health_score` directly to prevent circular logic.
2. Account-level stability
    * Features are aggregated across time to represent long-term behavior
3. Interpretability
    * Every feature can be explained without statistical jargon.
4. Business relevance
    * Each feature maps to a potential product or policy decision.
  
#### Selected Features & Rationale
| **Feature** | **Description** | **Business Meaning** |
|-------------|-----------------|----------------------|
|`avg_txn_count`| Mean monthly transactions | Engagement intensity |
|`avg_cashflow`| Mean net inflow-outflow | Income vs spending balance |
|`avg_balance`| Average balance | Financial buffer |
|`avg_volatility`| Average balance standard deviation | Stability vs financial stress |
|`avg_unique_merchants`| Mean merchant diversity | Lifestyle breadth |
|`low_balance_rate`| % months flagged low balance | Chronic liquidity risk |

* This set works because it covers engagement, liquidity, stability and behavioral diversity
* Avoids redundancy
* Avoids label leakage
* Works well with distance-based clustering

In [4]:
segmentation_df = df_health.groupby("Account No").agg(
    avg_txn_count=("txn_count", "mean"),
    avg_cashflow=("net_cash_flow", "mean"),
    avg_balance=("avg_balance", "mean"),
    avg_volatility=("balance_std", "mean"),
    avg_unique_merchants=("unique_merchants", "mean"),
    low_balance_rate=("low_balance_flag", "mean")
).reset_index()

segmentation_df.head()

,Account No,avg_txn_count,avg_cashflow,avg_balance,avg_volatility,avg_unique_merchants,low_balance_rate
0,100100738,4.250000,-318.331667,-2787.235833,468.764740,2.583333,1.0
1,100837224,2.363636,-235.625455,-5342.044545,200.300186,1.181818,1.0
2,101348775,15.333333,1074.268333,9560.719167,1147.342711,8.416667,0.0
3,103439190,19.166667,1867.390000,32648.592500,2493.829571,11.000000,0.0
4,104009728,13.833333,-1436.545833,14603.865000,1909.106936,9.750000,0.0


In [5]:
# Data Check
segmentation_df.isna().sum()

Account No              0
avg_txn_count           0
avg_cashflow            0
avg_balance             0
avg_volatility          1
avg_unique_merchants    0
low_balance_rate        0
dtype: int64

In [6]:
segmentation_df.fillna(0, inplace=True)
segmentation_df.isna().sum()

Account No              0
avg_txn_count           0
avg_cashflow            0
avg_balance             0
avg_volatility          0
avg_unique_merchants    0
low_balance_rate        0
dtype: int64

Missing values indicate lacck of activity rather than data errors, so zero is a meaningful replacement.

In [8]:
# Feature Distribution Review
segmentation_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Account No,1210.0,5.486037e+08,2.583069e+08,1.001007e+08,3.306048e+08,5.390146e+08,7.769432e+08,9.992808e+08
avg_txn_count,1210.0,7.822864e+00,4.214711e+00,1.000000e+00,4.750000e+00,7.000000e+00,9.916667e+00,2.491667e+01
avg_cashflow,1210.0,-1.629319e+02,9.653153e+02,-3.702743e+03,-8.995498e+02,-2.699737e+02,5.325625e+02,2.601263e+03
avg_balance,1210.0,-9.897687e+05,4.771598e+05,-1.892957e+06,-1.257680e+06,-1.115725e+06,-5.053380e+05,3.264859e+04
avg_volatility,1210.0,8.696807e+02,6.405925e+02,0.000000e+00,4.328794e+02,6.596387e+02,1.140378e+03,3.842932e+03
avg_unique_merchants,1210.0,4.315121e+00,2.390827e+00,1.000000e+00,2.500000e+00,3.666667e+00,5.333333e+00,1.341667e+01
low_balance_rate,1210.0,9.964463e-01,5.805447e-02,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


1. Average Transaction Count `avg_txn_count`
* Mean monthly transactions: ~8
* Wide range from 1 to ~25
* Right-skewed distribution

Accounts vary significantly in engagement intensity, making this a strong discriminator for segmentation.

2. Average Net Cash Flow `avg_cashflow`
* Mean value is negative, indicating overall net outflows
* Large variance, with both strong inflow and outflow accounts present

Cash flow patterns differ meaningfully across accounts, supporting its inclusion as a core behavioral dimension.

3. Average Balance `avg_balance`
* Predominantly negative balances across accounts
* Large spread across customers

Absolute balances are less informative than *relative differences* between users, reinforcing the need for normalization prior to clustering.

4. Average Balance Volatility `avg_volatility`
* Substantial spread (0 to ~3800)
* Indicates varying degrees of financial stability

Volatility captures stress patterns that are not visible through balances or cash flow alone, making it an essential feature.

5. Average Unique Merchants (`avg_unique_merchants`)
* Moderate variability
* Ranges from low merchant concentration to diverse spending behavior

Merchant diversity reflects lifestyle breadth and transactional variety, which is valuable for persona differentiation.

6. Low Balance Rate (`low_balance_rate`)
* Mean value $\approx$ 0.996
* Most accounts are flagged as low balance in nearly all months

This feature shows limited variance and therefore contributes minimal separation power in distance-based clustering.
However:
* It still encodes chronic liquidity risk
* Its effect will be naturally down-weighted after scaling
* It remains useful for post-cluster interpretation, even if not a primary driver

This feature is retained for interpretability while acknowledging its reduced clustering influence.

**Inference**
* Feature scales vary significantly (transactions vs cash flow vs balances)
* Several features are skewed or heavy tailed
* Distance-based clustering requires standardization

In [10]:
features = [
    "avg_txn_count",
    "avg_cashflow",
    "avg_balance",
    "avg_volatility",
    "avg_unique_merchants",
    "low_balance_rate"
]

X = segmentation_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
scaled_df = pd.DataFrame(X_scaled, columns=features)
scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
avg_txn_count,1210.0,1.123069e-16,1.000413,-1.619491,-0.729382,-0.195317,0.496990,4.057425
avg_cashflow,1210.0,2.348902e-17,1.000413,-3.668516,-0.763401,-0.110934,0.720782,2.864699
avg_balance,1210.0,4.697803e-17,1.000413,-1.893624,-0.561703,-0.264080,1.015657,2.143600
avg_volatility,1210.0,2.055289e-16,1.000413,-1.358181,-0.682153,-0.328023,0.422748,4.643328
avg_unique_merchants,1210.0,2.055289e-17,1.000413,-1.387173,-0.759516,-0.271338,0.426059,3.808434
low_balance_rate,1210.0,-1.127473e-15,1.000413,-17.171085,0.061239,0.061239,0.061239,0.061239


### Feature Scaling Validation
After applying StandardScaler, all selected features are centered around zero with unit variance. This confirms that no single feature will dominate distance calculations during clustering.

The low_balance_rate feature exhibits extreme negative standardized values due to its near-constant nature in the raw data. This is expected behavior and indicates that the scaler is functioning correctly rather than signaling a data issue.

With standardized inputs, the dataset is now suitable for distance-based clustering algorithms such as K-Means.

### Choosing the Optimal Number of Clusters
#### Objective
Determine the number of customer segments (`k`) that best balances:
* Cluster compactness
* Behavioral separation
* Business interpretability

We will use **two complementary methods**
1. Elbow Method - Measures how much additional explanatory power is gained by increasing `k`. 
2. Silhouette Score - Measures how well-separated clusters are:
    * +1 $\rightarrow$ well separated
    * 0 $\rightarrow$ overlapping
    * $<$ 0 $\rightarrow$ poor clustering

In [14]:
# Elbow Method
inertia = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

elbow_df = pd.DataFrame({
    "k": K_range,
    "inertia": inertia
})

elbow_df

,k,inertia
0,2,5240.936339
1,3,4051.031059
2,4,3331.037739
3,5,2867.481582
4,6,2463.536353
5,7,2233.967537
6,8,2099.078370


In [15]:
# Silhouette Score
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_df = pd.DataFrame({
    "k": K_range,
    "silhouette_score": silhouette_scores
})

silhouette_df

,k,silhouette_score
0,2,0.384823
1,3,0.383177
2,4,0.258178
3,5,0.270341
4,6,0.275304
5,7,0.246846
6,8,0.242502


Both the elbow method and silhouette analysis were used to determine the optimal number of clusters.

While k=2 achieved the highest silhouette score, it resulted in overly broad segmentation. k=3 provided nearly identical cluster separation while allowing for richer behavioral interpretation.

Based on this trade-off between statistical validity and business usability, k=3 was selected as the final number of customer segments.

In [17]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
segmentation_df["segment"] = kmeans.fit_predict(X_scaled)

segmentation_df["segment"].value_counts()

segment
1    921
0    285
2      4
Name: count, dtype: int64

In [18]:
# Inspecting Segment 2
segmentation_df[segmentation_df["segment"] == 2][features]

,avg_txn_count,avg_cashflow,avg_balance,avg_volatility,avg_unique_merchants,low_balance_rate
2,15.333333,1074.268333,9560.719167,1147.342711,8.416667,0.0
3,19.166667,1867.390000,32648.592500,2493.829571,11.000000,0.0
4,13.833333,-1436.545833,14603.865000,1909.106936,9.750000,0.0
5,6.500000,-850.272500,3282.097500,663.273082,4.833333,0.0


#### Outlier Cluster Identification
One cluster containing four accounts exhibits behavior that is structurally different from the rest of the population. These accounts maintain consistently positive balances, never trigger low balance flags, and show extremely high transaction activity and volatility.

This pattern indicates exceptional or non-representative usage rather than a stable customer persona. To preserve interpretability of the main segments, this cluster is treated as an outlier group and excluded from downstream persona analysis.

### Segment Profiling

In [20]:
core_segments = segmentation_df[segmentation_df["segment"] != 2]

segment_profile = (
    core_segments
    .groupby("segment")[features]
    .mean()
    .round(2)
    .reset_index()
)

segment_profile

,segment,avg_txn_count,avg_cashflow,avg_balance,avg_volatility,avg_unique_merchants,low_balance_rate
0,0,13.37,398.96,-1017481.80,1542.88,7.77,1.0
1,1,6.08,-338.23,-985556.84,658.39,3.23,1.0


#### Segment Summary
| **Segment** | **Size** | **Key Theme** |
|-------------|----------|---------------|
| Segment 0 | Smaller | High-activity, cash-positive but financially stressed |
| Segment 1 | Larger | Low-activity, cash-negative, structurally constrained |

Both segments show `low_balance_rate = 1.0`, meaning **every account consistently operates below the low-balance threshold.**

**Segment 0 - High Engagement but Financially Stressed**
* **Very high transaction activity** (avg_txn_count $\approx$ 13.4)
* **Net positive cashflow** (+399 on average)
* **Extremely volatile balances**
* High merchant diversity ($\approx$ 8 merchants)
* Persistently low balances

This segment actively uses the account and receives inflows, but money **moves out quickly**, preventing balance recovery. High volatility and merchant diversity indicate:
* Day-to-day spending behavior
* Possible paycheck-to-paycheck usage
* Limited savings buffer

**Product Implications**
* Budgeting and spend-insight tools
* Smart alerts for balance drops
* Short-term credit or overdraft protection
* Cashflow smoothing features

**Segment 1 - Low Engagement, Financially Constrained**
* **Low Transaction frequency** (avg_txn_count $\approx$ 6.1)
* **Net negative cashflow** (-388 on average)
* Low volatility
* Limited merchant diversity ($\approx$ 3 merchants)
* Persistently low balances

This segment shows **structural financial stress* rather than behavioral volatility. Users transact infrequently and rarely receive inflows, suggesting:
* Limited income sources
* Possibly inactive or secondary accounts
* Financial disengagement rather than overspending

**Product Implications**
* Re-engagement nudges
* Income discovery features
* Basic financial education
* Frictionless account usage incentives

In [39]:
segmentation_df.to_csv(
    "../outputs/account_segmentation.csv",
    index=False
)

cluster_centroids = (
    segmentation_df
    .groupby("segment")[features]
    .mean()
    .reset_index()
)

cluster_centroids.to_csv(
    "../outputs/cluster_centroids.csv"
)

elbow_df.to_csv("../outputs/k_elbow_results.csv", index=False)
silhouette_df.to_csv("../outputs/k_silhouette_results.csv", index=False)

segment_profile.to_csv("../outputs/segment_profile.csv")

In [23]:
import joblib

joblib.dump(scaler, "../models/segmentation_scaler.pkl")

['../models/segmentation_scaler.pkl']